# Task 4 Track A: Full HH-RLHF DPO Training (`align-dpo-hh`)

In [ ]:
!nvidia-smi
import os, shutil, pathlib

input_root = pathlib.Path('/kaggle/input')
src_dir = next(input_root.rglob('submission_adapter.py')).parent
work_repo = pathlib.Path('/tmp/repo')
if work_repo.exists():
    shutil.rmtree(work_repo)
shutil.copytree(src_dir, work_repo)
os.chdir(work_repo)
print('Found code dataset at:', src_dir)
print('COMMIT:', (work_repo / 'COMMIT.txt').read_text().strip())


In [ ]:
!python -m pip uninstall -y -q torchao || true
!python -m pip install -q "transformers>=4.43.0,<5" "datasets>=2.18,<5" huggingface_hub accelerate tqdm pyyaml sentencepiece safetensors
!python -m pip install -q -e . --no-deps
import torch, transformers, datasets
print('PyTorch:', torch.__version__, 'CUDA:', torch.cuda.is_available(), 'GPUs:', torch.cuda.device_count())
print('Transformers:', transformers.__version__, 'Datasets:', datasets.__version__)


In [ ]:
# Run full 15,000-pair HH-RLHF DPO training (469 steps, eval on 1,000 val pairs every 100 steps)
!PYTHONPATH=. python -m src.align.train_dpo --config configs/dpo_hh.yaml

# Sanity-check generation on 5 HH-RLHF prompts from the reloaded best/ checkpoint
!PYTHONPATH=. python -m src.align.generate \
  --model_name_or_path /kaggle/working/outputs/dpo_hh/best \
  --eval_set_path eval_sets/hh_rlhf_eval_300_seed42.json \
  --output_path /kaggle/working/outputs/dpo_hh/dpo_hh_sanity_5.json \
  --generator_name qwen2.5-0.5b-dpo-hh \
  --dataset_name hh_rlhf \
  --limit 5 \
  --batch_size 5
